In [1]:
import pandas as pd
import numpy as np
import math
import time
import warnings

# Suppress annoying SDV/Pandas warnings to keep the notebook output clean
warnings.filterwarnings('ignore')

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
def get_stratified_sample(df, target_col, sample_size):
    """
    Pandas-native Stratified Sampling function.
    Maintains exact proportions of the target column.
    """
    proportions = df[target_col].value_counts(normalize=True)
    pieces = []
    
    for val, prop in proportions.items():
        # Calculate how many rows of this specific class we need
        n_items = int(round(prop * sample_size))
        
        # Filter and randomly sample
        subset = df[df[target_col] == val]
        pieces.append(subset.sample(n=n_items, replace=False))
        
    # Combine and shuffle
    stratified_df = pd.concat(pieces).sample(frac=1).reset_index(drop=True)
    
    # Safety check: if rounding math caused us to be off by 1 or 2 rows, trim it to exact size
    if len(stratified_df) > sample_size:
        stratified_df = stratified_df.head(sample_size)
        
    return stratified_df

print("Sampling engine ready!")

Sampling engine ready!


In [4]:
print("======================================================")
print(" 🚀 BOOTSTRAPPED GAUSSIAN COPULA GENERATOR")
print("======================================================\n")

# --- 1. SETTINGS & INITIALIZATION ---
input_file = 'diabetes_binary_health_indicators_BRFSS2015.csv'
sampled_csv = 'sampled_data_accumulated.csv'
synthetic_csv = 'synthetic_data_accumulated.csv'

TOTAL_ITERATIONS = 100

print("1. Loading raw CDC data...")
raw_data = pd.read_csv(input_file)
n_total_rows = len(raw_data)

# Calculate sizes based on the math pattern
sample_size = int(math.sqrt(n_total_rows))
synthetic_size_per_loop = sample_size * 10

print(f"   -> Raw Data Rows (N): {n_total_rows:,}")
print(f"   -> Sample Size per loop (√N): {sample_size:,}")
print(f"   -> Synthetic Size per loop (√N * 10): {synthetic_size_per_loop:,}")
print(f"   -> Total Loops Planned: {TOTAL_ITERATIONS:,}\n")

# --- 2. METADATA DETECTION ---
print("2. Detecting Data Metadata (Only done once)...")
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(raw_data)
print("   -> Metadata locked in.\n")

# --- 3. THE MASTER LOOP ---
print(f"3. Starting {TOTAL_ITERATIONS} AI Training & Generation Loops...\n")

start_time_total = time.time()

for i in range(TOTAL_ITERATIONS):
    loop_start = time.time()
    
    # A. Take the Stratified Sample (Size = √N)
    current_sample = get_stratified_sample(raw_data, 'Diabetes_binary', sample_size)
    
    # B. Append to Sample CSV
    if i == 0:
        current_sample.to_csv(sampled_csv, index=False, mode='w')
    else:
        current_sample.to_csv(sampled_csv, index=False, mode='a', header=False)
        
    # C. Initialize a FRESH Copula and Train
    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(current_sample)
    
    # D. Generate Synthetic Data
    current_synthetic = synthesizer.sample(num_rows=synthetic_size_per_loop)
    
    # E. Append to Synthetic CSV
    if i == 0:
        current_synthetic.to_csv(synthetic_csv, index=False, mode='w')
    else:
        current_synthetic.to_csv(synthetic_csv, index=False, mode='a', header=False)

    # F. Console Progress & Time Estimation
    loop_duration = time.time() - loop_start
    
    if i == 0:
        estimated_total_seconds = loop_duration * TOTAL_ITERATIONS
        est_hours = estimated_total_seconds // 3600
        est_minutes = (estimated_total_seconds % 3600) // 60
        print(f"   [⏳ ESTIMATE] Based on Loop 1, this will take ~{int(est_hours)}h {int(est_minutes)}m to finish.")
        print("   ------------------------------------------------------")
        
    # Print progress every 10 loops so it doesn't crash the Jupyter Notebook output display
    if (i + 1) % 10 == 0 or i == 0:
        print(f"   -> Completed Loop {i + 1}/{TOTAL_ITERATIONS} | Accumulated Synthetic Rows: {(i + 1) * synthetic_size_per_loop:,}")

# --- 4. COMPLETION ---
total_time_mins = (time.time() - start_time_total) / 60
print("\n======================================================")
print(f" 🎉 EXPERIMENT COMPLETE!")
print(f" Time Taken: {total_time_mins:.2f} minutes")
print(f" Total Samples Pulled: {TOTAL_ITERATIONS * sample_size:,}")
print(f" Total Synthetic Rows Created: {TOTAL_ITERATIONS * synthetic_size_per_loop:,}")
print(f" Files Saved: '{sampled_csv}' and '{synthetic_csv}'")
print("======================================================")

 🚀 BOOTSTRAPPED GAUSSIAN COPULA GENERATOR

1. Loading raw CDC data...
   -> Raw Data Rows (N): 253,680
   -> Sample Size per loop (√N): 503
   -> Synthetic Size per loop (√N * 10): 5,030
   -> Total Loops Planned: 100

2. Detecting Data Metadata (Only done once)...
   -> Metadata locked in.

3. Starting 100 AI Training & Generation Loops...

   [⏳ ESTIMATE] Based on Loop 1, this will take ~0h 5m to finish.
   ------------------------------------------------------
   -> Completed Loop 1/100 | Accumulated Synthetic Rows: 5,030
   -> Completed Loop 10/100 | Accumulated Synthetic Rows: 50,300
   -> Completed Loop 20/100 | Accumulated Synthetic Rows: 100,600
   -> Completed Loop 30/100 | Accumulated Synthetic Rows: 150,900
   -> Completed Loop 40/100 | Accumulated Synthetic Rows: 201,200
   -> Completed Loop 50/100 | Accumulated Synthetic Rows: 251,500
   -> Completed Loop 60/100 | Accumulated Synthetic Rows: 301,800
   -> Completed Loop 70/100 | Accumulated Synthetic Rows: 352,100
   -> Co